# 06 – XGBoost trained and tuned on AWS SageMaker: Secondary Mushroom

**Worked on by:** Andreas Schellekens (setup, search space, evaluation; run in the AWS Academy lab)

> **GenAI disclosure:** the first version of this notebook was drafted with the help of Claude Code (AI assistant), based on the protocol of notebooks `03` to `05c`. All steps were reviewed by the team; we must be able to explain every cell ourselves.

## 1. Where this notebook runs, and what it does

**This notebook does not run on a laptop.** It runs in a SageMaker notebook inside an AWS Academy lab, started from Canvas. `README_AWS.md` (next to this notebook) gives every step. The lab notebook gets only two uploaded files: this notebook and `mushroom_aws_data.zip`, which `06_prepare_aws_upload.py` makes from the prepared files of `02_data_preparation.ipynb`.

The notebook itself trains nothing. It sends **jobs** to SageMaker, and SageMaker starts a separate machine for every job:

1. **Upload:** the training rows go to S3, the storage that SageMaker jobs read from. The validation and test sets never leave this notebook.
2. **Tuning:** a SageMaker **hyperparameter tuning job** runs 20 training jobs with the built-in XGBoost algorithm. Each job tries one combination of hyperparameters and reports its ROC-AUC on a hold-out part of the training set.
3. **Final training:** two more training jobs on all 3500 training rows, one with the best combination and one with XGBoost's default settings (to check whether tuning helped).
4. **Evaluation:** both models are downloaded and evaluated here, with the same protocol as every other model (`CLAUDE.md` 5.3): ROC-AUC on validation, a threshold for 90% recall on validation, and one evaluation on the test set.
5. **Download:** the results go into `mushroom_aws_results.zip`, which we take back to the repository for `07_model_comparison.ipynb`.

## 2. Setup

Imports, settings and constants. The first block contains the only **settings that may have to change in the lab** (README step 5):
- `INSTANCE_TYPE`: the machine type of every training job. A lab only allows some types; if the lab's own notebook uses another type for training, use that one.
- `MAX_JOBS` and `MAX_PARALLEL_JOBS`: 20 trials, 2 at a time (about 40 minutes). A lab limits how many machines may run at once; lower `MAX_PARALLEL_JOBS` to 1 if SageMaker refuses.
- `BUCKET` and `ROLE_ARN`: normally found automatically (section 3).
- `REUSE_TUNING_JOB`: only needed when the lab was reset during the search (README, troubleshooting).

We talk to AWS with `boto3`, the basic AWS library for Python, which is installed in every SageMaker notebook. The higher-level SageMaker Python SDK would hide these calls behind an `Estimator` and a `HyperparameterTuner`, but its interface changes between versions, and we do not know which version a lab has. With `boto3` every setting that goes to AWS is visible in the notebook.

In [ ]:
import json
import pickle
import subprocess
import sys
import tarfile
import time
import zipfile
from datetime import datetime
from importlib import metadata
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

# --- Settings to check in the lab (README_AWS.md, step 5) ---
INSTANCE_TYPE = "ml.m5.large"   # machine type of every training job
MAX_JOBS = 20                   # training jobs in the hyperparameter search
MAX_PARALLEL_JOBS = 2           # jobs at the same time; 1 if the lab refuses 2
BUCKET = None                   # None = SageMaker's default bucket (created if needed), or the name of the lab's bucket
ROLE_ARN = None                 # None = the role of this notebook, or the ARN of the lab's role (e.g. ...:role/LabRole)
REUSE_TUNING_JOB = None         # name of an earlier tuning job to continue with, after a lab reset

# --- Fixed choices: the same protocol as notebooks 03-05c ---
NOTEBOOK = "06_model_aws"
MODEL_NAME = "XGBoost on SageMaker (tuned)"
RANDOM_STATE = 42
TARGET = "is_poisonous"
RECALL_TARGET = 0.90
SPLITS = ["train", "validation", "test"]
EXPECTED_ROWS = {"train": 3500, "validation": 750, "test": 750}
XGBOOST_CONTAINER = "1.7-1"     # version of SageMaker's built-in XGBoost algorithm
XGBOOST_PIP = "1.7.6"           # the same XGBoost version, to load the trained models in this notebook
S3_PREFIX = "cloudai15/mushroom-xgboost"
POLL_SECONDS = 60

DATA_DIR = Path("Data")
UPLOAD_ZIP = Path("mushroom_aws_data.zip")
OUT_DIR = Path("outputs")       # everything we take back to the repository
RESULTS_ZIP = Path("mushroom_aws_results.zip")
JOB_NAMES_PATH = OUT_DIR / "job_names.json"
OUT_DIR.mkdir(exist_ok=True)


def data_path(split, kind="prepared"):
    # Same layout as in the repository: Data/<split>/mushroom_prepared_<split>.csv
    return DATA_DIR / split / f"mushroom_{kind}_{split}.csv"

## 3. Connecting to AWS

Three things are needed before any job can start:
- the **region** (AWS Academy labs use `us-east-1`, N. Virginia);
- the **role**: SageMaker jobs run under an IAM role that allows them to read and write S3. In a lab that role exists already (usually `LabRole`); we take the role of this notebook. A role given in `ROLE_ARN` wins;
- the **bucket**: an S3 bucket for the data and the trained models. SageMaker's usual default bucket is `sagemaker-<region>-<account>`; it is created if it does not exist yet. If the lab does not allow that, the error lists the buckets that do exist, and one of them goes into `BUCKET`.

The cell prints what it found. If this cell fails, nothing else will work; README_AWS.md (troubleshooting) lists the usual causes.

In [ ]:
session = boto3.session.Session()
region = session.region_name or "us-east-1"
account = session.client("sts").get_caller_identity()["Account"]
sm = session.client("sagemaker", region_name=region)
s3 = session.client("s3", region_name=region)


def find_role():
    if ROLE_ARN:
        return ROLE_ARN
    try:
        import sagemaker  # the SageMaker SDK knows the role of a notebook instance or Studio space
        return sagemaker.get_execution_role()
    except Exception:
        pass
    caller = session.client("sts").get_caller_identity()["Arn"]  # arn:aws:sts::<account>:assumed-role/<role>/<session>
    if ":assumed-role/" in caller:
        return f"arn:aws:iam::{account}:role/{caller.split(':assumed-role/')[1].split('/')[0]}"
    return f"arn:aws:iam::{account}:role/LabRole"  # the usual role in AWS Academy labs


def find_bucket():
    name = BUCKET or f"sagemaker-{region}-{account}"
    try:
        s3.head_bucket(Bucket=name)
        return name
    except ClientError:
        if BUCKET:
            raise  # a bucket given by hand must exist
    try:
        location = {} if region == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": region}}
        s3.create_bucket(Bucket=name, **location)
        return name
    except ClientError as error:
        try:
            existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
        except ClientError:
            existing = "(this role may not list buckets)"
        raise RuntimeError(f"Could not create bucket {name}: {error}. Set BUCKET to one of: {existing}") from error


role = find_role()
bucket = find_bucket()
print(f"Region:  {region}")
print(f"Role:    {role}")
print(f"Bucket:  s3://{bucket}/{S3_PREFIX}/")

## 4. Loading the data

The first time, the uploaded zip is unpacked into `Data/`, with the same folders as in the repository. These are the **prepared** files of notebook 02: 64 numerical columns (imputed, `log1p` + standardised, one-hot encoded by `models/mushroom_preprocessor.joblib`, fitted on train only) plus the target `is_poisonous` (1 = poisonous). SageMaker's built-in XGBoost only reads numbers, so the prepared files are the natural input: the cleaned files still contain text categories.

The checks make sure the lab got the same split as every other notebook: 3500 / 750 / 750 rows, 65 columns, no missing values, 284 poisonous mushrooms in validation and in test.

In [ ]:
if not data_path("train").exists():
    with zipfile.ZipFile(UPLOAD_ZIP) as archive:
        archive.extractall(DATA_DIR)

prepared = {split: pd.read_csv(data_path(split), index_col="row_id") for split in SPLITS}
for split, frame in prepared.items():
    assert len(frame) == EXPECTED_ROWS[split] and frame.shape[1] == 65, f"{split}: unexpected shape {frame.shape}"
    assert not frame.isna().any().any(), f"{split}: missing values"
assert prepared["validation"][TARGET].sum() == prepared["test"][TARGET].sum() == 284

X = {split: frame.drop(columns=TARGET) for split, frame in prepared.items()}
y = {split: frame[TARGET] for split, frame in prepared.items()}
feature_columns = list(X["train"].columns)
pd.DataFrame({split: {"rows": len(y[split]), "poisonous": int(y[split].sum()), "share poisonous": round(y[split].mean(), 3)}
              for split in SPLITS})

## 5. A tuning hold-out inside the training set

In notebooks 05a–05c every combination of hyperparameters was scored with **5-fold cross-validation** on the training set. A SageMaker tuning job works differently: every trial is one training job that reports one score on a `validation` **channel** (an S3 folder that the job evaluates on). Five folds would mean five jobs per trial, 100 jobs instead of 20, which does not fit in a lab session.

So we split the training set once, stratified, with the same seed as always: 80% (2800 rows) to fit each trial, 20% (700 rows) to score it. Two consequences:
- The **validation set stays out of the search**, exactly as in the other notebooks: it is needed later for the threshold and for comparing the models (07). Using it as SageMaker's "validation" channel would choose the hyperparameters and the threshold on the same rows.
- One hold-out of 700 rows is noisier than 5-fold CV (an AUC on about 700 rows has a standard error of about 0.015–0.02). The tuning AUCs below are therefore only used to rank the trials, not to compare with the CV scores of other notebooks. The comparable number is the validation AUC (section 10).

In [ ]:
X_fit, X_tune, y_fit, y_tune = train_test_split(X["train"], y["train"], test_size=0.2,
                                                stratify=y["train"], random_state=RANDOM_STATE)
print(f"fit:  {len(y_fit)} rows, {y_fit.mean():.3f} poisonous")
print(f"tune: {len(y_tune)} rows, {y_tune.mean():.3f} poisonous")

## 6. Uploading the training data to S3

SageMaker's XGBoost reads **CSV without a header, with the target in the first column** (that is how the algorithm knows which column to predict). We write three files and upload each to its own S3 folder (a channel points to a folder):
- `fit` and `tune`: the 80/20 split, for the tuning job;
- `train_full`: all 3500 training rows, for the final models.

The first characters of a file are printed as a check: the first value must be the target (0 or 1).

In [ ]:
def write_xgboost_csv(features, target, path):
    # Target first, no header, no index: the CSV format of SageMaker's built-in XGBoost
    pd.concat([target, features], axis=1).to_csv(path, header=False, index=False)


upload_dir = DATA_DIR / "s3_upload"
upload_dir.mkdir(exist_ok=True)
s3_uris = {}
for name, (features, target) in {"fit": (X_fit, y_fit), "tune": (X_tune, y_tune),
                                 "train_full": (X["train"], y["train"])}.items():
    path = upload_dir / f"{name}.csv"
    write_xgboost_csv(features, target, path)
    s3.upload_file(str(path), bucket, f"{S3_PREFIX}/data/{name}/{name}.csv")
    s3_uris[name] = f"s3://{bucket}/{S3_PREFIX}/data/{name}/"
    print(f"{name:10s} {len(target):5d} rows -> {s3_uris[name]}")

print("\nStart of fit.csv:", (upload_dir / "fit.csv").read_text()[:70], "...")

## 7. The algorithm: SageMaker's built-in XGBoost

**Why XGBoost?** It is the third big gradient-boosting library, next to LightGBM (PyCaret, 04) and scikit-learn's `HistGradientBoostingClassifier` (05b). Like those, it adds small trees one after the other, each one correcting the errors of the trees before it. It is one of SageMaker's **built-in algorithms**: AWS provides a ready container image, so we send only data and hyperparameters, no training code. The image is looked up for our region; the version (1.7-1) is fixed so that we can load the model later with the same XGBoost version.

The settings below are the same for every job:
- `objective = binary:logistic`: the model outputs a probability of poisonous, like `predict_proba` in the other notebooks;
- `eval_metric = auc`: the job computes ROC-AUC on the `validation` channel, which the tuning job maximises (the same ranking metric as the other notebooks);
- `seed = 42`; one `ml.m5.large` machine per job (2 CPUs, 8 GB, enough for 3500 rows); a job is stopped after 30 minutes at most.

**Paths not taken:**
- *Script mode* (our own scikit-learn script in a SageMaker container): possible, but a built-in algorithm is the standard way and needs no code upload.
- *SageMaker Autopilot / Canvas* (AutoML): we already have an AutoML notebook (04, PyCaret), and the assignment asks for a model that *we* tune.
- *An endpoint* (a hosted model that answers requests): costs money for every hour it runs and is not needed: we evaluate in this notebook. The deployment of the project uses the model from `07`.

In [ ]:
XGBOOST_IMAGE_ACCOUNTS = {  # AWS accounts that publish the XGBoost image, per region
    "us-east-1": "683313688378", "us-east-2": "257758044811", "us-west-1": "746614075791",
    "us-west-2": "246618743249", "eu-west-1": "141502667606", "eu-central-1": "492215442770",
}
try:
    from sagemaker import image_uris  # the SageMaker SDK, if this notebook has it
    image_uri = image_uris.retrieve("xgboost", region, XGBOOST_CONTAINER)
except Exception:
    image_uri = f"{XGBOOST_IMAGE_ACCOUNTS[region]}.dkr.ecr.{region}.amazonaws.com/sagemaker-xgboost:{XGBOOST_CONTAINER}"
print("XGBoost image:", image_uri)

STATIC_HYPERPARAMETERS = {"objective": "binary:logistic", "eval_metric": "auc", "seed": str(RANDOM_STATE)}
JOB_SETTINGS = {  # the parts of a training job that are the same for every job
    "AlgorithmSpecification": {"TrainingImage": image_uri, "TrainingInputMode": "File"},
    "RoleArn": role,
    "OutputDataConfig": {"S3OutputPath": f"s3://{bucket}/{S3_PREFIX}/models/"},
    "ResourceConfig": {"InstanceType": INSTANCE_TYPE, "InstanceCount": 1, "VolumeSizeInGB": 5},
    "StoppingCondition": {"MaxRuntimeInSeconds": 1800},
}


def channel(name, s3_uri):
    # One input folder of a training job: "train" to fit on, "validation" to score on
    return {"ChannelName": name, "ContentType": "text/csv", "CompressionType": "None",
            "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": s3_uri,
                                            "S3DataDistributionType": "FullyReplicated"}}}

## 8. The search space and the tuning job

| Hyperparameter | Range (scale) | What it controls | Counterpart in 05b (HGB) |
|---|---|---|---|
| `eta` | 0.01 to 0.3 (log) | Learning rate: how much every new tree corrects | `learning_rate` |
| `num_round` | 50 to 800 (log) | Number of trees | chosen by early stopping in 05b |
| `max_depth` | 2 to 10 | Depth of each tree | `max_leaf_nodes`, `max_depth` |
| `min_child_weight` | 1 to 20 (log) | Minimum "weight" of rows per leaf (for this objective about the number of rows times p(1-p)); higher = no leaves for a few, possibly mislabelled rows | `min_samples_leaf` |
| `subsample` | 0.5 to 1 | Share of the rows each tree sees | (none) |
| `colsample_bytree` | 0.4 to 1 | Share of the columns each tree sees | `max_features` |
| `lambda` | 0.01 to 10 (log) | L2 penalty that shrinks the leaf values | `l2_regularization` |

**Why `num_round` is searched here.** In 05b early stopping chose the number of trees, using 10% of the rows as an inner hold-out. The final model here is trained on *all* training rows, without a hold-out to stop on, so the number of trees has to be a fixed hyperparameter, chosen together with the learning rate (a small `eta` needs many trees). No class weights: 05b found none were needed, and the threshold handles the 90% recall target.

**Bayesian search instead of random search.** 05b tried 40 random combinations (`RandomizedSearchCV`). SageMaker's default strategy is **Bayesian optimisation**: after each finished trial, it updates a model of "AUC as a function of the hyperparameters" and picks the next combination where that model expects a high AUC or is still uncertain. Every trial learns from the earlier ones, so 20 trials go further than 20 random ones. With 2 jobs at a time it chooses 2 combinations while the others run. `RandomSeed` makes the choices repeatable.

The tuning job is created **only once**: its name is saved in `outputs/job_names.json`. If the notebook is run again (for example after the browser lost its connection), it continues with the same job instead of starting a second search.

In [ ]:
def continuous(name, low, high, scale="Linear"):
    return {"Name": name, "MinValue": str(low), "MaxValue": str(high), "ScalingType": scale}


def integer(name, low, high, scale="Linear"):
    return {"Name": name, "MinValue": str(low), "MaxValue": str(high), "ScalingType": scale}


INTEGER_HYPERPARAMETERS = ["num_round", "max_depth"]
PARAMETER_RANGES = {
    "ContinuousParameterRanges": [
        continuous("eta", 0.01, 0.3, "Logarithmic"),
        continuous("min_child_weight", 1, 20, "Logarithmic"),
        continuous("subsample", 0.5, 1.0),
        continuous("colsample_bytree", 0.4, 1.0),
        continuous("lambda", 0.01, 10, "Logarithmic"),
    ],
    "IntegerParameterRanges": [
        integer("num_round", 50, 800, "Logarithmic"),
        integer("max_depth", 2, 10),
    ],
    "CategoricalParameterRanges": [],
}


def job_names():
    return json.loads(JOB_NAMES_PATH.read_text()) if JOB_NAMES_PATH.exists() else {}


def save_job_name(key, name):
    JOB_NAMES_PATH.write_text(json.dumps({**job_names(), key: name}, indent=2))


tuning_job_name = REUSE_TUNING_JOB or job_names().get("tuning")
if tuning_job_name:
    print("Continuing with the existing tuning job:", tuning_job_name)
else:
    tuning_job_name = f"mushroom-xgb-{datetime.now():%m%d-%H%M%S}"  # at most 32 characters
    sm.create_hyper_parameter_tuning_job(
        HyperParameterTuningJobName=tuning_job_name,
        HyperParameterTuningJobConfig={
            "Strategy": "Bayesian",
            "HyperParameterTuningJobObjective": {"Type": "Maximize", "MetricName": "validation:auc"},
            "ResourceLimits": {"MaxNumberOfTrainingJobs": MAX_JOBS, "MaxParallelTrainingJobs": MAX_PARALLEL_JOBS},
            "ParameterRanges": PARAMETER_RANGES,
            "TrainingJobEarlyStoppingType": "Off",
            "RandomSeed": RANDOM_STATE,
        },
        TrainingJobDefinition={
            **JOB_SETTINGS,
            "StaticHyperParameters": STATIC_HYPERPARAMETERS,
            "InputDataConfig": [channel("train", s3_uris["fit"]), channel("validation", s3_uris["tune"])],
        },
    )
    print("Started tuning job:", tuning_job_name)
save_job_name("tuning", tuning_job_name)

The cell below waits until the search is finished and prints a line whenever something changes (about 40 minutes with 20 jobs, 2 at a time: each job needs a few minutes, mostly to start its machine). The jobs run in AWS, not in this notebook: closing the browser does not stop them. The same progress is visible in the AWS console under *SageMaker AI > Training > Hyperparameter tuning jobs*.

In [ ]:
def wait_for_tuning(name):
    last_line = None
    while True:
        job = sm.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=name)
        status, counts = job["HyperParameterTuningJobStatus"], job["TrainingJobStatusCounters"]
        best = job.get("BestTrainingJob", {}).get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value")
        line = (f"{status}: {counts.get('Completed', 0)} completed, {counts.get('InProgress', 0)} running, "
                f"{counts.get('RetryableError', 0) + counts.get('NonRetryableError', 0)} failed"
                + (f", best AUC so far {best:.4f}" if best is not None else ""))
        if line != last_line:
            print(f"{datetime.now():%H:%M}  {line}")
            last_line = line
        if status in ("Completed", "Stopped", "Failed"):
            return job
        time.sleep(POLL_SECONDS)


tuning = wait_for_tuning(tuning_job_name)
if "BestTrainingJob" not in tuning:
    raise RuntimeError(f"No trial succeeded ({tuning.get('FailureReason', 'no reason given')}). "
                       "Open a failed training job in the console to see its error.")

## 9. The results of the search

Every trial with its hyperparameters and its ROC-AUC on the 700 tuning rows, in the order in which they started. The table is also saved as `outputs/mushroom_xgboost_tuning_jobs.csv`.

In [ ]:
summaries = sm.list_training_jobs_for_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name, MaxResults=100)["TrainingJobSummaries"]
trials = pd.DataFrame([{
    "job": s["TrainingJobName"],
    "status": s["TrainingJobStatus"],
    "start": s.get("TrainingStartTime"),
    "tuning_auc": s.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value"),
    **{name: float(value) for name, value in s["TunedHyperParameters"].items()},
} for s in summaries])
trials[INTEGER_HYPERPARAMETERS] = trials[INTEGER_HYPERPARAMETERS].astype("Int64")
trials = trials.sort_values("start", na_position="last").reset_index(drop=True)
trials.index = pd.RangeIndex(1, len(trials) + 1, name="trial")
trials.to_csv(OUT_DIR / "mushroom_xgboost_tuning_jobs.csv")

print(f"{len(trials)} trials, {trials['tuning_auc'].notna().sum()} with a score")
trials.drop(columns=["job", "start"]).sort_values("tuning_auc", ascending=False).round(4)

Left: the AUC of each trial in start order, and the best AUC so far. If the Bayesian search learns, the later trials are good more often than the early, exploratory ones. Right: the learning rate against the AUC, coloured by tree depth.

In [ ]:
scored = trials.dropna(subset=["tuning_auc"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scored.index, scored["tuning_auc"], "o", color="#264653", label="trial")
axes[0].plot(scored.index, scored["tuning_auc"].cummax(), color="#e76f51", label="best so far")
axes[0].set(xlabel="trial (start order)", ylabel="ROC-AUC on the tuning rows", title="Bayesian search, trial by trial")
axes[0].legend()
points = axes[1].scatter(scored["eta"], scored["tuning_auc"], c=scored["max_depth"], cmap="viridis", s=45)
axes[1].set(xscale="log", xlabel="eta (learning rate, log scale)", ylabel="ROC-AUC on the tuning rows",
            title="Learning rate vs. AUC")
fig.colorbar(points, ax=axes[1], label="max_depth")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_job = tuning["BestTrainingJob"]
best_hyperparameters = best_job["TunedHyperParameters"]
best_tuning_auc = best_job["FinalHyperParameterTuningJobObjectiveMetric"]["Value"]
print(f"Best trial: {best_job['TrainingJobName']}, ROC-AUC on the tuning rows {best_tuning_auc:.4f}")
pd.Series(best_hyperparameters, name="best value")

**Interpretation** (to be written after the run, with the numbers above): how far apart are the best and the worst trials? Do the later trials score better than the early ones? Which learning rates and depths win? Is the best combination at the edge of a range (then the range was too narrow)?

## 10. Final training on all training rows: tuned and default

The best trial was fitted on 80% of the training rows. As `RandomizedSearchCV` does in the other notebooks (`refit`), the final model is trained again with the best hyperparameters on **all 3500 training rows**. At the same time a second job trains XGBoost with its **default settings** (100 trees, `eta` 0.3, `max_depth` 6), because the protocol checks on the validation set whether tuning helped at all (05b: it barely did).

Both jobs run in parallel and get only a `train` channel: there is nothing to score on during training. Like the tuning job, they are started only once (names in `outputs/job_names.json`).

In [ ]:
DEFAULT_HYPERPARAMETERS = {"num_round": "100"}  # everything else at XGBoost's defaults
final_hyperparameters = {"tuned": {**STATIC_HYPERPARAMETERS, **best_hyperparameters},
                         "default": {**STATIC_HYPERPARAMETERS, **DEFAULT_HYPERPARAMETERS}}

for kind, hyperparameters in final_hyperparameters.items():
    key = f"final_{kind}"
    if key in job_names():
        print(f"{kind}: already started as {job_names()[key]}")
        continue
    name = f"mushroom-xgb-{kind}-{datetime.now():%m%d-%H%M%S}"
    sm.create_training_job(TrainingJobName=name, HyperParameters=hyperparameters,
                           InputDataConfig=[channel("train", s3_uris["train_full"])], **JOB_SETTINGS)
    save_job_name(key, name)
    print(f"{kind}: started {name}")

Waiting for both jobs (a few minutes). `BillableTimeInSeconds` is the machine time AWS charges for.

In [ ]:
def wait_for_training(names):
    last_line = None
    while True:
        jobs = {name: sm.describe_training_job(TrainingJobName=name) for name in names}
        line = " | ".join(f"{name}: {job['TrainingJobStatus']} ({job['SecondaryStatus']})" for name, job in jobs.items())
        if line != last_line:
            print(f"{datetime.now():%H:%M}  {line}")
            last_line = line
        if all(job["TrainingJobStatus"] in ("Completed", "Failed", "Stopped") for job in jobs.values()):
            return jobs
        time.sleep(POLL_SECONDS)


final_jobs = wait_for_training([job_names()["final_tuned"], job_names()["final_default"]])
for name, job in final_jobs.items():
    if job["TrainingJobStatus"] != "Completed":
        raise RuntimeError(f"{name}: {job['TrainingJobStatus']} - {job.get('FailureReason')}")
    print(f"{name}: billable time {job.get('BillableTimeInSeconds')} s")

## 11. Downloading the models

Every job saved its model as `model.tar.gz` in S3. We download both, and load them with the **same XGBoost version** as the container (1.7.6), so the saved file format matches; it is installed here if this notebook has another version or none. The tuned model is saved as `outputs/mushroom_xgboost_sagemaker.tar.gz`: this file goes to the repository.

Inside the archive is one file, `xgboost-model`: the trees in XGBoost's own format (containers before version 1.3 used a Python pickle instead, which the fallback handles).

In [ ]:
try:
    installed = metadata.version("xgboost")
except metadata.PackageNotFoundError:
    installed = None
if installed != XGBOOST_PIP:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"xgboost=={XGBOOST_PIP}"])
import xgboost as xgb

print("XGBoost in this notebook:", xgb.__version__)


def download_model(job_name, filename):
    uri = sm.describe_training_job(TrainingJobName=job_name)["ModelArtifacts"]["S3ModelArtifacts"]
    model_bucket, key = uri.removeprefix("s3://").split("/", 1)
    path = OUT_DIR / filename
    s3.download_file(model_bucket, key, str(path))
    return path


def load_booster(tar_path):
    with tarfile.open(tar_path) as archive:
        member = next(m for m in archive.getmembers() if m.name.endswith("xgboost-model"))
        model_bytes = archive.extractfile(member).read()
    booster = xgb.Booster()
    try:
        booster.load_model(bytearray(model_bytes))  # XGBoost's own format (containers 1.3 and later)
    except xgb.core.XGBoostError:
        booster = pickle.loads(model_bytes)  # older containers saved a pickle
    return booster


def predict(booster, features):
    # binary:logistic -> the probability of poisonous, like predict_proba(...)[:, 1]
    return booster.predict(xgb.DMatrix(features.to_numpy()))


model_files = {"tuned": download_model(job_names()["final_tuned"], "mushroom_xgboost_sagemaker.tar.gz"),
               "default": download_model(job_names()["final_default"], "mushroom_xgboost_sagemaker_default.tar.gz")}
boosters = {kind: load_booster(path) for kind, path in model_files.items()}
for kind, path in model_files.items():
    print(f"{kind}: {path} ({path.stat().st_size / 1e3:.0f} KB), {boosters[kind].num_boosted_rounds()} trees")

## 12. Validation check: did tuning help?

Both models score the validation set, which played no part in the search. The scores of the other notebooks are listed for context (from `models/metrics.csv`, not recomputed here). As in 05b the tuned model stays the model of this notebook, whatever this check says: the choice was made in the search, and the validation set is not used to undo it. The check tells us how much the search was worth.

In [ ]:
p_val = {kind: predict(booster, X["validation"]) for kind, booster in boosters.items()}
validation_auc = pd.Series({
    "XGBoost on SageMaker, tuned": roc_auc_score(y["validation"], p_val["tuned"]),
    "XGBoost on SageMaker, default": roc_auc_score(y["validation"], p_val["default"]),
    "05c stacked ensemble (reference)": 0.821,
    "05b gradient boosting, deployed (reference)": 0.819,
    "05a random forest (reference)": 0.817,
    "03 logistic regression (reference)": 0.704,
}, name="validation ROC-AUC")
validation_auc.round(3)

**Interpretation** (to be written after the run): tuned vs. default on validation; where XGBoost lands among the other tree models (a difference of 0.01 is within the noise of 750 rows, `07` compares the models paired on the same rows).

## 13. Threshold for 90% recall

Exactly as in notebooks 05a–05c: the tuned model (trained on the training rows) predicts the validation set, and we take the highest threshold that still catches 90% of the poisonous validation mushrooms. A mushroom is called poisonous when its probability is at least this threshold.

In [ ]:
def threshold_for_recall(y_true, p, target):
    # Highest threshold whose recall is still >= target (recall only drops when the threshold rises)
    _, recall, thresholds = precision_recall_curve(y_true, p)
    return thresholds[recall[:-1] >= target].max()


threshold = float(threshold_for_recall(y["validation"], p_val["tuned"], RECALL_TARGET))
pred = p_val["tuned"] >= threshold
print(f"Threshold: {threshold:.3f}")
print(f"Validation recall {recall_score(y['validation'], pred):.3f}, precision {precision_score(y['validation'], pred):.3f}, "
      f"edible mushrooms flagged: {(pred & (y['validation'] == 0)).sum()} of {(y['validation'] == 0).sum()}")

## 14. Validation and test evaluation

The tuned model is evaluated **once** on the test set, at threshold 0.5 and at the chosen threshold, with the same `evaluate` helper as notebooks 03–05c; the validation rows are listed for `07`. The rows are saved as `outputs/mushroom_xgboost_metrics.csv`, with the columns of `models/metrics.csv`, and are added to that file in the repository.

In [ ]:
def evaluate(name, split, y_true, p_poisonous, threshold=0.5):
    y_pred = (p_poisonous >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "model": name,
        "notebook": NOTEBOOK,
        "split": split,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "recall_poisonous": recall_score(y_true, y_pred),
        "precision_poisonous": precision_score(y_true, y_pred, zero_division=0),
        "f1_poisonous": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p_poisonous),
        "missed_poisonous": fn,
        "false_alarms": fp,
    }


p_test = predict(boosters["tuned"], X["test"])
results = []
for split, p_split in [("validation", p_val["tuned"]), ("test", p_test)]:
    results.append(evaluate(MODEL_NAME, split, y[split], p_split, threshold=0.5))
    results.append(evaluate(f"XGBoost on SageMaker (tuned, recall {RECALL_TARGET:.2f})", split, y[split], p_split,
                            threshold=threshold))
pd.DataFrame(results).to_csv(OUT_DIR / "mushroom_xgboost_metrics.csv", index=False)
pd.DataFrame(results).set_index(["model", "split"]).drop(columns="notebook").round(3)

**Interpretation** (to be written after the run): test AUC, recall and the number of missed poisonous mushrooms at the threshold, compared with the deployed gradient boosting model (05b: test AUC 0.847, 17 missed / 301 false alarms of 284 / 466).

## 15. Saving the results

Everything we take back to the repository goes into `outputs/` and then into one zip file, `mushroom_aws_results.zip`:

| File | Content | Goes to (repository) |
|---|---|---|
| `mushroom_xgboost_sagemaker.tar.gz` | The tuned model as SageMaker saved it | `models/` |
| `mushroom_xgboost_sagemaker.json` | Threshold, hyperparameters, job names, input columns | `models/` |
| `mushroom_xgboost_metrics.csv` | Validation and test rows | added to `models/metrics.csv` |
| `mushroom_xgboost_predictions.csv` | Probability per `row_id` for validation and test | check that `07` loads the model correctly |
| `mushroom_xgboost_tuning_jobs.csv` | All trials of the search | `models/` (for the report) |
| `mushroom_xgboost_sagemaker_default.tar.gz`, `job_names.json` | Default model, job names | not needed in the repository |

The model takes the 64 **prepared** columns, so in the repository it is used behind `models/mushroom_preprocessor.joblib` (input: the 11 cleaned columns, like every other model).

In [ ]:
model_info = {
    "model": MODEL_NAME,
    "notebook": NOTEBOOK,
    "threshold": round(threshold, 4),
    "threshold_rule": f"highest threshold with recall (poisonous) >= {RECALL_TARGET} on the validation set",
    "tuning_holdout_roc_auc": round(float(best_tuning_auc), 4),
    "validation_roc_auc": round(float(roc_auc_score(y["validation"], p_val["tuned"])), 4),
    "validation_roc_auc_default": round(float(roc_auc_score(y["validation"], p_val["default"])), 4),
    "hyperparameters": {name: int(value) if name in INTEGER_HYPERPARAMETERS else float(value)
                        for name, value in best_hyperparameters.items()},
    "static_hyperparameters": STATIC_HYPERPARAMETERS,
    "n_trees": int(boosters["tuned"].num_boosted_rounds()),
    "model_file": model_files["tuned"].name,
    "xgboost_version": XGBOOST_PIP,
    "sagemaker": {"region": region, "image": image_uri, "instance_type": INSTANCE_TYPE, "strategy": "Bayesian",
                  "max_jobs": MAX_JOBS, "max_parallel_jobs": MAX_PARALLEL_JOBS,
                  "tuning_job": tuning_job_name, "final_training_job": job_names()["final_tuned"]},
    "input": "the 64 prepared columns (output of models/mushroom_preprocessor.joblib), in this order",
    "input_columns": feature_columns,
}
(OUT_DIR / "mushroom_xgboost_sagemaker.json").write_text(json.dumps(model_info, indent=2))

pd.concat([pd.DataFrame({"row_id": X[split].index, "split": split, "p_poisonous": p})
           for split, p in [("validation", p_val["tuned"]), ("test", p_test)]]
          ).to_csv(OUT_DIR / "mushroom_xgboost_predictions.csv", index=False)

with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.iterdir()):
        archive.write(path, arcname=path.name)
print(f"{RESULTS_ZIP} ({RESULTS_ZIP.stat().st_size / 1e3:.0f} KB):")
for path in sorted(OUT_DIR.iterdir()):
    print(f"  {path.name:45s} {path.stat().st_size / 1e3:8.1f} KB")

**Now download two files** (right-click in the file browser on the left > *Download*): `mushroom_aws_results.zip` and this notebook, after saving it (*File > Save*). README_AWS.md, step 8, says where they go in the repository.

## 16. Cleaning up

This notebook starts no endpoints, and training jobs stop by themselves, so nothing keeps costing money once the jobs are done. The cell checks that no training job is still running and no endpoint exists.

The data and models in S3 cost almost nothing (a few MB). They can be deleted by setting `CLEAN_UP = True`, **but only after both files are downloaded**: the models in S3 are then gone for good.

In [ ]:
CLEAN_UP = False  # True only after mushroom_aws_results.zip and this notebook are downloaded

running = sm.list_training_jobs(StatusEquals="InProgress")["TrainingJobSummaries"]
print("Training jobs still running:", [job["TrainingJobName"] for job in running] or "none")
print("Endpoints:", [endpoint["EndpointName"] for endpoint in sm.list_endpoints()["Endpoints"]] or "none")

if CLEAN_UP:
    keys = [{"Key": item["Key"]}
            for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=f"{S3_PREFIX}/")
            for item in page.get("Contents", [])]
    for start in range(0, len(keys), 1000):
        s3.delete_objects(Bucket=bucket, Delete={"Objects": keys[start:start + 1000]})
    print(f"Deleted {len(keys)} objects under s3://{bucket}/{S3_PREFIX}/")

## 17. Conclusion

(To be written after the run: the best hyperparameters, tuned vs. default on validation, test results at the threshold, and how XGBoost on SageMaker compares with the deployed gradient boosting model. The paired comparison with all other models follows in `07_model_comparison.ipynb`.)

**What this notebook shows about the cloud workflow:** the data goes to S3, SageMaker runs every trial on its own machine with a ready-made algorithm, the tuning job chooses the next trials with Bayesian optimisation, and the final model comes back as a file that we evaluate with exactly the same protocol as the models trained on a laptop.